# MKM411 — Computational Fluid Dynamics
## Lecture: Scientific Machine Learning for Fluid Dynamics
### Part 1 of 2 — Neural Networks & the SciML Landscape
---
**Department of Mechanical & Aeronautical Engineering | University of Pretoria**  
*Prof K.J. Craig & Prof M. Bhamjee*


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import warnings
warnings.filterwarnings('ignore')

from lecture_utils import (
    UP_BLUE, UP_GOLD, ACCENT, SEED,
    SmallNet, DemoANN,
    draw_network,
    plot_feedforward,
    plot_xavier_initialisation,
    plot_backpropagation,
    plot_bias_variance,
    plot_sciml_landscape,
    activation_explorer,
    forward_pass_widget,
    loss_landscape_widget,
    plot_deeponet, plot_fno, plot_gnn, plot_gan,
    plot_rnn, plot_reservoir_computing, plot_foundation_models,
    plot_method_comparison,
)

torch.manual_seed(SEED)
np.random.seed(SEED)
print("Environment ready.")


---
## 1. Motivation — Why is Machine Learning Entering CFD?

Classical CFD methods (FDM, FVM, FEM) are mature, reliable, and physically grounded.
So why are researchers increasingly turning to machine learning?

| Challenge | Classical CFD | Implication |
|-----------|--------------|-------------|
| **High-dimensional parameter spaces** | Re-solve for every new set | Design optimisation requires thousands of solves |
| **Real-time prediction** | Minutes to hours per solve | Not viable for control or digital twins |
| **Noisy or incomplete data** | Needs clean BCs and ICs | Experimental data is messy |
| **Inverse problems** | Hard to formulate | Cannot easily infer unknown fields |

### What ML offers
- **Surrogates** — fast approximations of expensive solvers
- **Data assimilation** — learn from experimental observations
- **Generalisation** — one model predicts across a parameter space
- **Inverse problem solving** — infer unknowns from observable data

### What ML does *not* replace
Physics. A purely data-driven model trained on limited data will violate
conservation laws and fail to generalise. This motivates **Scientific Machine
Learning** — embedding physical knowledge into the learning process.


In [ ]:
# Curse of dimensionality — how quickly does a parameter sweep grow?
dims = np.arange(1, 9)
points_per_dim = 10

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(dims, points_per_dim**dims, color=UP_BLUE, alpha=0.8)
ax.set_yscale('log')
ax.set_xlabel('Number of parameters', fontsize=12)
ax.set_ylabel('Solver runs required', fontsize=12)
ax.set_title('Curse of Dimensionality — 10 values per parameter',
             fontsize=12, color=UP_BLUE)
ax.set_xticks(dims)
ax.grid(True, alpha=0.3, axis='y')
for d, label in {1: 'vary rho', 3: 'vary rho,cp,k', 6: 'vary 6 params'}.items():
    ax.text(d, points_per_dim**d * 2, label, ha='center',
            fontsize=8, color='gray')
plt.tight_layout(); plt.show()
print(f"6 parameters x 10 values = {10**6:,} solver runs")
print(f"At 1 min/solve: {10**6/60/24:.0f} days of compute")


---
## 2. Artificial Neural Networks — Architecture

A neural network is a parametric function $f_\theta: \mathbb{R}^n \to \mathbb{R}^m$
composed of layers of simple operations.

### The fully-connected layer — matrix form

$$\mathbf{z}^{(l)} = W^{(l)}\mathbf{a}^{(l-1)} + \mathbf{b}^{(l)}$$

$$\mathbf{a}^{(l)} = \sigma\!\left(\mathbf{z}^{(l)}\right)$$

where $W^{(l)} \in \mathbb{R}^{n_l \times n_{l-1}}$, $\mathbf{b}^{(l)} \in \mathbb{R}^{n_l}$,
and $\sigma$ is the **activation function** applied element-wise.

### Universal approximation theorem
A network with a single hidden layer and sufficient width can approximate
any continuous function on a compact domain to arbitrary precision.
In practice, **depth** is more parameter-efficient than width alone.

### The project network
Input: $(\hat{x}, \hat{y}, \hat{t}, \hat{\rho}, \hat{c}_p, \hat{k})$ — 6 inputs  
Architecture: 5 hidden layers × 40 neurons — **6,881 parameters**  
Output: $T$ [K]


In [ ]:
fig = plot_feedforward()
plt.show()


---
## 3. Activation Functions

Activation functions introduce **non-linearity**. Without them, a deep network
collapses to a single linear transformation — regardless of depth.

### The PINN constraint
PINNs compute $\partial^2 T / \partial x^2$ via autograd through the network.
The activation must be **at least twice differentiable** with **non-zero second
derivative** — otherwise the PDE residual is identically zero everywhere.

**`tanh` satisfies this. `ReLU` does not** — its second derivative is zero almost everywhere.


In [ ]:
activation_explorer()


---
## 3b. Weight Initialisation — Getting the Scale Right

Initialising weights randomly seems obvious. But the *scale* of that randomness
matters critically for deep networks.

### Why not zeros?
If all weights are zero, every neuron computes the same output and receives
the same gradient — the network never breaks symmetry and all neurons remain
identical throughout training. **Random initialisation is essential for symmetry breaking.**

### The vanishing/exploding gradient problem
Consider a signal $\mathbf{a}^{(0)}$ passed through $L$ layers with weights
drawn from $\mathcal{N}(0, \sigma^2)$:

$$\text{Var}(\mathbf{z}^{(l)}) = n_{l-1} \cdot \sigma^2 \cdot \text{Var}(\mathbf{a}^{(l-1)})$$

- If $\sigma^2$ is too small — variance shrinks at each layer → **vanishing signal**
- If $\sigma^2$ is too large — variance grows at each layer → **exploding signal**

### Xavier / Glorot initialisation
Choose $\sigma^2$ to preserve variance across layers:

$$\text{Var}(w) = \frac{2}{n_{\text{in}} + n_{\text{out}}}$$

This ensures $\text{Var}(\mathbf{z}^{(l)}) \approx \text{Var}(\mathbf{a}^{(l-1)})$ — stable signal propagation.

In PyTorch: `nn.init.xavier_normal_(layer.weight)` — used throughout the project.


In [ ]:
fig = plot_xavier_initialisation()
plt.show()


---
## 4. Training — Learning from Data

### Loss function (data-driven ANN)
$$\mathcal{L}_{\text{ANN}}(\theta) = \frac{1}{N} \sum_{i=1}^{N} \left( f_\theta(\mathbf{x}_i) - T_i \right)^2$$

### Backpropagation — the chain rule through the network

Define the **error signal** at layer $l$:

$$\boldsymbol{\delta}^{(l)} = \frac{\partial \mathcal{L}}{\partial \mathbf{z}^{(l)}}$$

**Output layer** ($L$):
$$\boldsymbol{\delta}^{(L)} = \nabla_{\mathbf{a}^{(L)}} \mathcal{L} \odot \sigma'(\mathbf{z}^{(L)})$$

**Hidden layers** — propagate backwards:
$$\boldsymbol{\delta}^{(l)} = \left(W^{(l+1)}\right)^T \boldsymbol{\delta}^{(l+1)} \odot \sigma'(\mathbf{z}^{(l)})$$

**Weight gradients**:
$$\nabla_{W^{(l)}} \mathcal{L} = \boldsymbol{\delta}^{(l)} \left(\mathbf{a}^{(l-1)}\right)^T$$

This is computed automatically by **autograd** in PyTorch — the same mechanism
that computes PDE derivatives in PINNs.


In [ ]:
fig = plot_backpropagation()
plt.show()


### Adam Optimiser

Standard gradient descent: $\theta \leftarrow \theta - \eta \nabla_\theta \mathcal{L}$

**Adam** adapts the learning rate per parameter using moment estimates:

$$m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t \qquad \text{(first moment — mean)}$$
$$v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2 \qquad \text{(second moment — uncentred variance)}$$
$$\theta_t = \theta_{t-1} - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

Adam is **robust to poorly scaled problems** and is the default optimiser for PINNs.


In [ ]:
# Interactive forward pass widget
forward_pass_widget()


In [ ]:
# Interactive loss landscape — take gradient steps, adjust learning rate
loss_landscape_widget()


---
## 4b. Mini-Batch Training

### Why not use all data at once?

**Full-batch gradient descent** computes the true gradient but:
- Requires all data in memory simultaneously
- Finds the nearest local minimum — may not generalise well
- Very slow per update on large datasets

**Stochastic gradient descent (SGD)** uses one sample per update:
- Fast updates but very noisy
- High variance in gradient estimates

**Mini-batch** (standard practice) balances both:
- Sample a random subset of $B$ points each epoch
- Gradient estimate has controlled noise — helps escape local minima
- Efficient on GPU (vectorised matrix operations)

### In the project
The ANN DataLoader uses `batch_size = 512`. Each epoch shuffles all
training data and processes it in batches of 512. The PINN uses full-batch
Adam on its collocation points — since there is no data, only physics residuals.

| Method | Gradient | Memory | Generalisation |
|--------|---------|--------|---------------|
| Full batch | Exact | High | Risk of sharp minima |
| SGD | Noisy | Low | Often better generalisation |
| Mini-batch | Approximate | Controlled | Best trade-off |


---
## 4c. Overfitting and the Bias-Variance Trade-off

A model that performs well on training data but poorly on unseen data has **overfit**.
It has memorised the training set rather than learning the underlying pattern.

### Bias-variance decomposition

For any predictor, the expected test error decomposes as:

$$\mathbb{E}\left[(y - f_\theta(x))^2\right] = \underbrace{\text{Bias}^2}_{\text{underfitting}} + \underbrace{\text{Variance}}_{\text{overfitting}} + \underbrace{\sigma^2_{\epsilon}}_{\text{irreducible noise}}$$

- **High bias** — model too simple, cannot capture the pattern
- **High variance** — model too complex, fits noise in training data
- **Optimal** — balance between the two

### How to detect overfitting
Monitor **training loss vs validation loss**:
- Both decreasing → still learning
- Train decreasing, val increasing → overfitting
- Both high → underfitting

### Mitigations
- **More data** — most effective
- **Dropout** — randomly zero neurons during training
- **L2 regularisation** — penalise large weights: $\mathcal{L} \leftarrow \mathcal{L} + \lambda \|\theta\|^2$
- **Early stopping** — stop when validation loss increases


In [ ]:
fig = plot_bias_variance()
plt.show()


---
## 5. The SciML Landscape — A Panorama

**Scientific Machine Learning** embeds physical knowledge into the learning process.
Below is a tour of the main approaches, each with its own architectural idea
and niche in the fluid dynamics workflow.


In [ ]:
fig = plot_sciml_landscape(highlight_pinn=True)
plt.show()


### 5.1 DeepONet — Learning Operators Between Function Spaces

In [ ]:
fig = plot_deeponet()
plt.show()


### 5.2 Fourier Neural Operator (FNO) — Convolution in Spectral Space

In [ ]:
fig = plot_fno()
plt.show()


### 5.3 Graph Neural Networks (GNNs) — Learning on Meshes

In [ ]:
fig = plot_gnn()
plt.show()


### 5.4 Generative Adversarial Networks (GANs) — Learning Distributions

In [ ]:
fig = plot_gan()
plt.show()


### 5.5 Recurrent Neural Networks (RNNs / LSTMs) — Temporal Memory

In [ ]:
fig = plot_rnn()
plt.show()


### 5.6 Reservoir Computing — Fixed Dynamics, Trained Readout

Reservoir Computing takes a fundamentally different approach to the other methods.
The reservoir is a fixed, randomly connected recurrent network — it is never trained.
Only the output layer is learned, via simple linear regression.

The physics interpretation is elegant: the reservoir acts as a **nonlinear dynamical
projection** of the input time series into a high-dimensional state space. The output
layer then reads off the relevant coordinates. Echo State Networks (ESNs) are the most
common implementation.

This makes RC extremely fast to train compared to deep RNNs, while retaining good
performance on chaotic and nonlinear dynamical systems — exactly the regime of
turbulent flows.


In [ ]:
fig = plot_reservoir_computing()
plt.show()


### 5.7 Foundation Models — Pre-trained at Scale

The most recent frontier. Large transformer-based models pre-trained on massive
simulation and observational datasets, then fine-tuned for specific tasks.
Analogous to how GPT is pre-trained on text then fine-tuned per application.

Currently the most prominent examples are in weather and climate modelling,
where training data is abundant and the governing equations are well-established.
Application to general CFD is an active area of research.


In [ ]:
fig = plot_foundation_models()
plt.show()


### Summary — Method Comparison

In [ ]:
fig = plot_method_comparison()
plt.show()


---
## Summary — Part 1

| Concept | Key takeaway |
|---------|-------------|
| **Motivation** | Classical CFD is expensive; ML offers fast surrogates and generalisation |
| **Architecture** | Layers of matrix ops + non-linear activations — 6,881 parameters in the project |
| **Activation** | Must be twice differentiable for PINNs — use `tanh`, not `ReLU` |
| **Initialisation** | Xavier preserves signal variance — prevents vanishing/exploding gradients |
| **Backpropagation** | Chain rule through the network — same mechanism as PINN autograd |
| **Mini-batches** | Balance between noisy SGD and expensive full-batch — standard in practice |
| **Overfitting** | Monitor train vs val loss — stop early or regularise |
| **SciML landscape** | Seven methods from PINNs to Foundation Models — each with a niche |

**In Part 2:** We embed the Navier-Stokes equations into the loss function
and apply this to the Von Karman vortex street behind a cylinder.
